## 📊 Part 3 — PCA for High-Dimensional Business Data: Retail Transactions

Real customer analytics often involves dozens of features: purchase frequency across categories,
recency scores, channel preferences, seasonal indices. PCA compresses these into a handful
of interpretable dimensions.

In [ ]:
# Synthetic retail customer feature matrix
# 500 customers × 20 features: category spend, frequency, recency, channel mix
np.random.seed(7)
n_customers = 500

# Underlying latent dimensions
price_sensitivity  = np.random.normal(0, 1, n_customers)  # PC1
engagement         = np.random.normal(0, 1, n_customers)  # PC2
channel_pref       = np.random.normal(0, 1, n_customers)  # PC3

features = {
    'grocery_spend':     2*engagement + price_sensitivity*0.5 + np.random.normal(0,.5,n_customers),
    'electronics_spend': engagement - price_sensitivity + np.random.normal(0,.6,n_customers),
    'clothing_spend':    engagement*0.8 + channel_pref*0.4 + np.random.normal(0,.5,n_customers),
    'home_spend':        engagement*0.6 + np.random.normal(0,.7,n_customers),
    'beauty_spend':      engagement*0.7 - channel_pref*0.3 + np.random.normal(0,.4,n_customers),
    'frequency':         engagement*1.2 + np.random.normal(0,.4,n_customers),
    'recency_days':      -engagement*0.8 + np.random.normal(0,.5,n_customers),
    'avg_basket':        engagement*0.5 - price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'returns_rate':      price_sensitivity*0.3 + np.random.normal(0,.4,n_customers),
    'promo_sensitivity': price_sensitivity*1.5 + np.random.normal(0,.3,n_customers),
    'mobile_pct':        channel_pref*1.2 + np.random.normal(0,.4,n_customers),
    'email_open_rate':   engagement*0.4 + channel_pref*0.5 + np.random.normal(0,.5,n_customers),
    'loyalty_points':    engagement*1.0 + np.random.normal(0,.4,n_customers),
    'weekend_pct':       channel_pref*0.3 + np.random.normal(0,.6,n_customers),
    'private_label_pct': price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'cross_category':    engagement*0.9 + np.random.normal(0,.4,n_customers),
    'seasonal_index':    np.random.normal(0,1,n_customers),
    'store_vs_online':   channel_pref*1.0 + np.random.normal(0,.4,n_customers),
    'referral_flag':     engagement*0.3 + np.random.normal(0,.8,n_customers),
    'complaint_count':  -engagement*0.4 + price_sensitivity*0.2 + np.random.normal(0,.5,n_customers),
}
X_retail = pd.DataFrame(features)
print(f"Retail customer feature matrix: {X_retail.shape}")
print(f"\n20 features per customer — hard to visualize or model directly")
print(f"PCA will find the key dimensions driving customer variation")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize (features are on different scales — spend vs rates vs counts)
scaler_r = StandardScaler()
X_scaled_r = scaler_r.fit_transform(X_retail)

# Fit PCA
pca_retail = PCA()
X_pca_r = pca_retail.fit_transform(X_scaled_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
cumvar = np.cumsum(pca_retail.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1

axes[0].bar(range(1,21), pca_retail.explained_variance_ratio_*100,
            color='#1e5fa8', edgecolor='white', alpha=0.8, label='Individual')
axes[0].plot(range(1,21), cumvar*100, 'o-', color='#e85d20', lw=2,
             markersize=5, label='Cumulative')
axes[0].axhline(90, color='#888', lw=1, ls='--', alpha=0.6, label='90% threshold')
axes[0].axvline(n_90, color='#1a7a45', lw=1.5, ls='--',
                label=f'{n_90} PCs explain 90%')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('How Many Dimensions Capture Retail Customer Variation?')
axes[0].legend(fontsize=8)

# PC1 vs PC2 scatter — colored by engagement
axes[1].scatter(X_pca_r[:,0], X_pca_r[:,1],
                c=features['frequency'], cmap='RdYlGn', s=20, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_retail.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca_retail.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('Customers in 2D PCA Space\n(color = purchase frequency)')
sm = plt.cm.ScalarMappable(cmap='RdYlGn')
plt.colorbar(sm, ax=axes[1], label='Purchase Frequency', shrink=0.8)

plt.tight_layout(); plt.show()
print(f"\n\u2714 {n_90} components capture 90% of customer variation (vs 20 original features)")
print(f"   Compression ratio: {20/n_90:.1f}x — huge simplification for downstream modeling")

In [ ]:
# What does each PC represent? — Loadings analysis
loadings = pd.DataFrame(pca_retail.components_[:3].T,
                        index=X_retail.columns,
                        columns=['PC1','PC2','PC3'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pc in zip(axes, ['PC1','PC2','PC3']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#e85d20' if v > 0 else '#1e5fa8' for v in sorted_load]
    ax.barh(sorted_load.index, sorted_load.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{pc} Loadings\n({pca_retail.explained_variance_ratio_[int(pc[-1])-1]*100:.1f}% variance)')
    ax.set_xlabel('Loading')

plt.suptitle('PCA Loadings: What Does Each Component Represent?\n'
             'Orange = positive contribution, Blue = negative',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()
print("\n\u2714 PC1: high loadings on frequency, loyalty, cross-category = 'Engagement'")
print("   PC2: high loadings on promo sensitivity, private label = 'Price Sensitivity'")
print("   PC3: high loadings on mobile, store_vs_online = 'Channel Preference'")
print("   These match the latent dimensions we built into the data!")

In [ ]:
answers = {
    "q1": "",  # What does the first principal component maximize?
    "q2": "",  # What do loadings tell you about a principal component?
    "q3": "",  # Why must features be standardized before PCA?
    "q4": "",  # You have 100 features. The first 5 PCs explain 85% of variance. What would you do next?
    "q5": "",  # What is one limitation of PCA for classification tasks?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "PCA"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")
